# Free-Living Gait Analysis (.cwa files)
This notebook is independent of the main script, but **reuses its functions** by importing them from `gait_lib.py` (shared module).

Unlike the protocol-based dataset, there is no `patients.csv` with labeled activities here -- the complete recording of each patient is analyzed. The files are `.cwa` (Axivity binary format), which are read with `skdh.io.ReadCwa` instead of `pandas.read_csv`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install -q scikit-digital-health


In [ ]:
import os
import sys
import pandas as pd

# Carpeta donde guardaste gait_lib.py (ajusta si lo pones en otro sitio)
sys.path.append('/content/drive/MyDrive/dataset')

from gait_lib import build_gait_old, find_patient_files, run_gait_freeliving


## 2. Configuration

In [ ]:
FREE_LIVING_DIR = "/content/drive/MyDrive/dataset_pd/free_living"
OUTPUT_DIR       = "/content/drive/MyDrive/Lasige/FreeLiving_PD"
os.makedirs(OUTPUT_DIR, exist_ok=True)

GROUP = "PD"          # change to "Control" if you are running the control free_living folder
MAX_PATIENTS = 15     # test limit; increase the number once you confirm it works
PATIENT_HEIGHT_M = 1.69


## 3. Locate trunk files (.cwa)

File names are not 100% consistent (`trunk-`, `trunkfreelivi`, `trunl-`...), so we first list what was found for review before running anything computationally intensive.

In [ ]:
trunk_files, skipped, posibles_erratas = find_patient_files(
    FREE_LIVING_DIR, keyword="trunk", fuzzy_keyword="trun"
)

print(f"Patients with a detected trunk file: {len(trunk_files)}")
for pid, path in list(trunk_files.items())[:10]:
    print(f"  {pid} -> {os.path.basename(path)}")

# Warning if any 'trunk' file found is NOT a .cwa (in case mixed formats are present in the folder)
no_cwa = {pid: p for pid, p in trunk_files.items() if not p.lower().endswith('.cwa')}
if no_cwa:
    print(f"\n[AVISO] {len(no_cwa)} trunk file(s) are NOT .cwa (they will be read as .csv):")
    for pid, p in no_cwa.items():
        print(f"  {pid} -> {os.path.basename(p)}")

print(f"\nFicheros ignorados (no contienen 'trunk'): {len(skipped)}")

if posibles_erratas:
    print("\n[REVISAR] These files contain 'trun' but not 'trunk' "
          "(posibles erratas de nombre, ej. 'trunl-'):")
    for name in posibles_erratas:
        print(f"  {name}")


If names marked **[REVIEW]** appear in the previous cell, decide what to do with them (rename them in Drive or manually add them to `trunk_files` below) before continuing.

```python
# Example of manually adding a file with an unusual name:
# trunk_files["30837"] = "/content/drive/MyDrive/dataset_pd/free_living/30837-trunl-....cwa"
```

## 4. (Optional) Test reading ONE .cwa file before running everything

`.cwa` files can contain recordings spanning several days -- it is worth confirming that they are read correctly and checking how many hours of data they contain before launching the full analysis.

In [ ]:
from gait_lib import read_recording

test_pid = list(trunk_files.keys())[0]
t, acc, recording_start = read_recording(trunk_files[test_pid])
print(f"Patient {test_pid}")
print(f"Inicio de grabacion: {recording_start}")
print(f"Duration: {t[-1]/3600:.1f} horas")
print(f"Muestras: {len(t)}")


## 5. Run gait analysis for each patient

In [ ]:
gait = build_gait_old()  # same configuration que en Script2.ipynb

all_summaries = []
processed = 0

for patient_id, file_path in trunk_files.items():
    if processed >= MAX_PATIENTS:
        break

    print(f"\n[{patient_id}] processing {os.path.basename(file_path)}")
    summary_df = run_gait_freeliving(
        gait_model=gait,
        patient_id=patient_id,
        file_path=file_path,
        output_dir=OUTPUT_DIR,
        group=GROUP,
        height_m=PATIENT_HEIGHT_M,
        fs=None,
    )
    if summary_df is not None:
        all_summaries.append(summary_df)
        processed += 1

print(f"\n{processed} patient(s) processed")


## 6. Combine all summaries into a single CSV

In [ ]:
if all_summaries:
    combined = pd.concat(all_summaries, ignore_index=True)
    combined_path = os.path.join(OUTPUT_DIR, "ALL_patients_freeliving_summary.csv")
    combined.to_csv(combined_path, index=False)
    print(f"Combined summary saved -> {combined_path}")
    display(combined)
else:
    print("No summary was generated.")
